# US Equities Liquidity Intelligence

## Fragmentation, venue concentration and share shifts in 2025

This short study uses public Cboe market-volume data to build a repeatable view of the US equity liquidity landscape. It is designed as a **market-structure and programme-management analysis**: validate the source, identify material changes, and translate them into questions for venue, broker and execution discussions.

**Scope:** daily consolidated activity by market centre and listing tape. **Not in scope:** order-level fills, spreads, price improvement, markouts or latency.


### Questions

1. How much reported activity occurs on exchange versus through FINRA Trade Reporting Facilities?
2. Which exchanges are most relevant for Tape A, B and C securities?
3. Does the venue mix change on the highest-activity sessions?
4. Which venue-share changes would merit follow-up in a liquidity review?


In [1]:
from pathlib import Path
from urllib.request import urlopen
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import seaborn as sns

YEAR = 2025
BASE_URL = (
    "https://cdn.cboe.com/resources/us/equities/"
    "market-statistics/historical-market-volume"
)
DATA_PATH = Path("../data/raw/cboe") / f"market_history_{YEAR}.csv"

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_rows", 30)
pd.set_option("display.float_format", "{:.4f}".format)


In [2]:
def download_cboe_volume(year: int, destination: Path) -> Path:
    """Download once, then use the local cache on later runs."""
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return destination

    url = f"{BASE_URL}/market_history_{year}.csv"
    with urlopen(url, timeout=30) as response:
        destination.write_bytes(response.read())
    return destination


source_path = download_cboe_volume(YEAR, DATA_PATH)
raw = pd.read_csv(source_path, parse_dates=["Day"])

print(f"Rows: {len(raw):,}")
print(f"Trading days: {raw['Day'].nunique()}")
print(f"Coverage: {raw['Day'].min():%d %b %Y} to {raw['Day'].max():%d %b %Y}")
print(f"Market participants: {raw['Market Participant'].nunique()}")


Rows: 4,805
Trading days: 250
Coverage: 02 Jan 2025 to 31 Dec 2025
Market participants: 20


### Data controls

The venue map is explicit rather than inferred from names. The checks below make a new or renamed venue visible, prevent duplicate venue-day records, and reconcile the three tape notionals to the published total. That matters because an apparently successful chart can still be wrong when a source schema changes.


In [3]:
classification = {
    '24X National Exchange (G)': 'Exchange',
    'BYX Equities (Y)': 'Exchange', 'BZX Equities (Z)': 'Exchange',
    'EDGA Equities (J)': 'Exchange', 'EDGX Equities (K)': 'Exchange',
    'Investors Exchange (V)': 'Exchange', 'LTSE (L)': 'Exchange',
    'MIAX Pearl (H)': 'Exchange', 'Members Exchange (U)': 'Exchange',
    'NASDAQ (Q)': 'Exchange', 'NASDAQ BX (B)': 'Exchange',
    'NASDAQ PSX (X)': 'Exchange', 'NYSE (N)': 'Exchange',
    'NYSE American (A)': 'Exchange', 'NYSE Arca (P)': 'Exchange',
    'NYSE National (C)': 'Exchange', 'NYSE Texas (M)': 'Exchange',
    'FINRA / NYSE TRF (DN)': 'Off-exchange (TRF)',
    'FINRA / Nasdaq TRF Carteret (DQ)': 'Off-exchange (TRF)',
    'FINRA / Nasdaq TRF Chicago (DB)': 'Off-exchange (TRF)',
}

required = {
    'Day', 'Market Participant', 'Tape A Notional', 'Tape B Notional',
    'Tape C Notional', 'Total Notional', 'Total Shares', 'Total Trade Count'
}
assert required.issubset(raw.columns), "Required columns are missing"
assert not raw.duplicated(['Day', 'Market Participant']).any(), "Duplicate venue-day rows"

unmapped = sorted(set(raw['Market Participant']) - set(classification))
assert not unmapped, f"Update the venue map: {unmapped}"

tape_total = raw[['Tape A Notional', 'Tape B Notional', 'Tape C Notional']].sum(axis=1)
assert np.allclose(tape_total, raw['Total Notional'], rtol=1e-10, atol=0.05)

data = raw.copy()
data['Venue Type'] = data['Market Participant'].map(classification)
print("PASS: schema, uniqueness, venue coverage and notional reconciliation")


PASS: schema, uniqueness, venue coverage and notional reconciliation


## 1. Start with the market-wide liquidity split

Notional value is the primary lens because it better reflects the value of risk transferred than share count alone. I retain shares and trade count as cross-checks because the conclusion changes with the chosen denominator.


In [4]:
venue_type = data.pivot_table(
    index='Day', columns='Venue Type', values='Total Notional',
    aggfunc='sum', fill_value=0
).sort_index()
venue_type['Total Notional'] = venue_type.sum(axis=1)
venue_type['Off-exchange share'] = (
    venue_type['Off-exchange (TRF)'] / venue_type['Total Notional']
)
venue_type['20-day average'] = venue_type['Off-exchange share'].rolling(20, min_periods=10).mean()

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
axes[0].plot(venue_type.index, venue_type['Total Notional'] / 1e9, color='#12263A')
axes[0].fill_between(venue_type.index, venue_type['Total Notional'] / 1e9, color='#12263A', alpha=.10)
axes[0].set_ylabel('Notional ($bn)')
axes[0].set_title('US equity trading activity and off-exchange share, 2025', loc='left', weight='bold')

axes[1].plot(venue_type.index, venue_type['Off-exchange share'], color='#8492A6', alpha=.7, label='Daily')
axes[1].plot(venue_type.index, venue_type['20-day average'], color='#0E8A83', linewidth=2.5, label='20-day average')
axes[1].axhline(venue_type['Off-exchange share'].mean(), color='#F28E5B', linestyle='--', label='2025 mean')
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].set_ylabel('Off-exchange share')
axes[1].legend(frameon=False, ncol=3)
plt.tight_layout()
plt.show()


![US equity notional and off-exchange share](../assets/images/us_market_overview.png)

In [5]:
metric_comparison = []
for measure in ['Total Notional', 'Total Shares', 'Total Trade Count']:
    split = data.pivot_table(
        index='Day', columns='Venue Type', values=measure, aggfunc='sum', fill_value=0
    )
    off_exchange_share = split['Off-exchange (TRF)'] / split.sum(axis=1)
    metric_comparison.append({
        'Activity measure': measure.replace('Total ', ''),
        'Average off-exchange share': off_exchange_share.mean(),
        'Daily minimum': off_exchange_share.min(),
        'Daily maximum': off_exchange_share.max(),
    })

metric_comparison = pd.DataFrame(metric_comparison)
metric_display = metric_comparison.copy()
for column in ['Average off-exchange share', 'Daily minimum', 'Daily maximum']:
    metric_display[column] = metric_display[column].map('{:.1%}'.format)
metric_display


Activity measure,Average off-exchange share,Daily minimum,Daily maximum
Notional,45.0%,37.9%,48.9%
Shares,50.7%,40.4%,54.7%
Trade Count,41.1%,33.5%,45.3%


**Read-through.** Off-exchange activity averaged **45.0% of daily notional**, compared with **50.7% of shares** and **41.1% of trades**. The gap is economically useful: it suggests the off-exchange channel handles a different mix of security prices and trade sizes. Aggregate data cannot identify the underlying client or execution mechanism, so this is a follow-up question rather than a causal conclusion.


## 2. Venue relevance depends on the listing tape

An aggregate venue ranking can hide where an exchange is genuinely strong. The heatmap compares each exchange's share of **on-exchange notional within each tape**.


In [6]:
exchange = data.loc[data['Venue Type'].eq('Exchange')]
tape_columns = ['Tape A Notional', 'Tape B Notional', 'Tape C Notional']
tape_share = exchange.groupby('Market Participant')[tape_columns].sum()
tape_share = tape_share.div(tape_share.sum(axis=0), axis=1)
tape_share.columns = ['Tape A', 'Tape B', 'Tape C']

leaders = tape_share.sum(axis=1).nlargest(8).index
tape_display = tape_share.loc[leaders].copy()
tape_display.loc['Other exchanges'] = tape_share.drop(leaders).sum()

fig, ax = plt.subplots(figsize=(10.5, 7.5))
sns.heatmap(tape_display * 100, annot=True, fmt='.1f', cmap='BuGn',
            linewidths=.7, cbar_kws={'label': 'Share of on-exchange notional (%)'}, ax=ax)
ax.set_title('Venue leadership differs sharply by listing tape', loc='left', weight='bold')
ax.set_xlabel(''); ax.set_ylabel('')
plt.tight_layout(); plt.show()


![On-exchange venue share by listing tape](../assets/images/us_tape_structure.png)

**Read-through.** The three largest exchanges represented **68.6% of on-exchange notional**, but leadership is tape-specific: NYSE leads Tape A, NYSE Arca leads Tape B and Nasdaq leads Tape C. For venue coverage or connectivity decisions, the security universe matters more than a single consolidated market-share number.


## 3. High-activity sessions form a different liquidity regime

I label the top 10% of sessions by total notional and compare them with the other 90%. A bootstrap resamples days within each group to quantify uncertainty in the mean difference. This is descriptive, not causal: calendar events, index rebalances and auctions may contribute to the result.


In [7]:
exchange_notional = exchange.pivot_table(
    index='Day', columns='Market Participant', values='Total Notional',
    aggfunc='sum', fill_value=0
).sort_index()
exchange_share = exchange_notional.div(exchange_notional.sum(axis=1), axis=0)

venue_type['Exchange HHI'] = exchange_share.pow(2).sum(axis=1) * 10_000
venue_type['Effective venue count'] = 10_000 / venue_type['Exchange HHI']
high_volume = venue_type['Total Notional'] >= venue_type['Total Notional'].quantile(.90)

def bootstrap_difference(values, group, n_boot=20_000, seed=7):
    high = values[group].to_numpy(); other = values[~group].to_numpy()
    rng = np.random.default_rng(seed)
    differences = np.empty(n_boot)
    for i in range(n_boot):
        differences[i] = (
            rng.choice(high, len(high), replace=True).mean()
            - rng.choice(other, len(other), replace=True).mean()
        )
    return high.mean() - other.mean(), np.quantile(differences, [.025, .975])

off_diff, off_ci = bootstrap_difference(venue_type['Off-exchange share'], high_volume)
hhi_diff, hhi_ci = bootstrap_difference(venue_type['Exchange HHI'], high_volume)

print(f"Off-exchange share difference: {off_diff*100:+.2f} pp "
      f"(bootstrap 95% CI {off_ci[0]*100:+.2f}, {off_ci[1]*100:+.2f})")
print(f"Exchange HHI difference: {hhi_diff:+.0f} "
      f"(bootstrap 95% CI {hhi_ci[0]:+.0f}, {hhi_ci[1]:+.0f})")

regime = pd.DataFrame({
    'Session': np.where(high_volume, 'Top-decile notional', 'Other sessions'),
    'Off-exchange share': venue_type['Off-exchange share'],
    'Exchange HHI': venue_type['Exchange HHI'],
})
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
sns.boxplot(data=regime, x='Session', y='Off-exchange share', color='#0E8A83', ax=axes[0])
sns.boxplot(data=regime, x='Session', y='Exchange HHI', color='#F28E5B', ax=axes[1])
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[0].set_title('Off-exchange share', weight='bold')
axes[1].set_title('On-exchange concentration', weight='bold')
for ax in axes: ax.set_xlabel('')
plt.tight_layout(); plt.show()


Off-exchange share difference: -3.34 pp (bootstrap 95% CI -4.23, -2.48)
Exchange HHI difference: +188 (bootstrap 95% CI +80, +313)


![Liquidity mix on high-activity and other sessions](../assets/images/us_activity_regimes.png)

In [8]:
regime_summary = pd.DataFrame({
    'Metric': ['Off-exchange share', 'Exchange HHI', 'Effective venue count'],
    'Other sessions': [
        venue_type.loc[~high_volume, 'Off-exchange share'].mean(),
        venue_type.loc[~high_volume, 'Exchange HHI'].mean(),
        venue_type.loc[~high_volume, 'Effective venue count'].mean(),
    ],
    'Top-decile sessions': [
        venue_type.loc[high_volume, 'Off-exchange share'].mean(),
        venue_type.loc[high_volume, 'Exchange HHI'].mean(),
        venue_type.loc[high_volume, 'Effective venue count'].mean(),
    ],
})
regime_summary


Metric,Other sessions,Top-decile sessions
Off-exchange share,0.452876,0.419467
Exchange HHI,1937.657925,2125.601170
Effective venue count,5.167269,4.782767


**Read-through.** On the 25 highest-notional sessions, off-exchange share was **3.34 percentage points lower** and exchange HHI was **188 points higher** than on other days. The effective exchange count also fell. This is consistent with peak activity becoming more concentrated in displayed/auction venues, but confirming the mechanism would require intraday auction flags and order-level execution data.


## 4. Turn the research into a monitoring output

Two views are useful together: a slower quarter-on-quarter comparison for the liquidity roadmap, and a daily signal against the prior 20 sessions for investigation. The signal is not an alert by itself; it prioritises where to ask what changed in flow, fee tiers, routing, auctions or venue functionality.


In [9]:
quarterly = exchange_share.groupby(exchange_share.index.to_period('Q')).mean().T
share_shift = pd.DataFrame({
    'Q1 share': quarterly[pd.Period('2025Q1')],
    'Q4 share': quarterly[pd.Period('2025Q4')],
})
share_shift['Change (pp)'] = (share_shift['Q4 share'] - share_shift['Q1 share']) * 100

material = pd.concat([
    share_shift.nsmallest(5, 'Change (pp)'),
    share_shift.nlargest(5, 'Change (pp)')
]).sort_values('Change (pp)')

fig, ax = plt.subplots(figsize=(10.5, 6.5))
colors = np.where(material['Change (pp)'] < 0, '#F28E5B', '#0E8A83')
ax.barh(material.index, material['Change (pp)'], color=colors)
ax.axvline(0, color='#12263A', linewidth=1)
ax.set_xlabel('Change in average daily on-exchange share (percentage points)')
ax.set_title('Largest venue-share changes: Q4 versus Q1 2025', loc='left', weight='bold')
plt.tight_layout(); plt.show()


![Largest venue-share changes between Q1 and Q4](../assets/images/us_share_shifts.png)

In [10]:
long_share = exchange_share.stack().rename('Daily share').reset_index()
long_share = long_share.sort_values(['Market Participant', 'Day'])
grouped = long_share.groupby('Market Participant')['Daily share']
long_share['Prior 20D mean'] = grouped.transform(
    lambda values: values.shift(1).rolling(20, min_periods=15).mean()
)
long_share['Prior 20D volatility'] = grouped.transform(
    lambda values: values.shift(1).rolling(20, min_periods=15).std()
)
long_share['Change (pp)'] = (
    long_share['Daily share'] - long_share['Prior 20D mean']
) * 100
long_share['z-score'] = (
    (long_share['Daily share'] - long_share['Prior 20D mean'])
    / long_share['Prior 20D volatility']
)

latest_monitor = long_share.loc[long_share['Day'].eq(long_share['Day'].max())]
latest_monitor = latest_monitor.sort_values(
    'Change (pp)', key=lambda values: values.abs(), ascending=False
)
monitor_display = latest_monitor.head(6).set_index('Market Participant')[
    ['Daily share', 'Prior 20D mean', 'Change (pp)', 'z-score']
].copy()
monitor_display['Daily share'] = monitor_display['Daily share'].map('{:.2%}'.format)
monitor_display['Prior 20D mean'] = monitor_display['Prior 20D mean'].map('{:.2%}'.format)
monitor_display['Change (pp)'] = monitor_display['Change (pp)'].map('{:+.2f}'.format)
monitor_display['z-score'] = monitor_display['z-score'].map('{:+.2f}'.format)
monitor_display


,Daily share,Prior 20D mean,Change (pp),z-score
Market Participant,,,,
NYSE Arca (P),16.92%,15.86%,+1.07,+0.58
NASDAQ (Q),36.11%,36.98%,-0.87,-0.43
NYSE (N),18.05%,17.37%,+0.69,+0.23
Investors Exchange (V),5.49%,6.12%,-0.63,-0.89
NYSE Texas (M),1.24%,1.58%,-0.34,-0.83
BZX Equities (Z),7.29%,7.59%,-0.30,-0.39


## What I would take to a liquidity forum

- **Market context:** off-exchange activity is structurally material, but its measured share depends on whether the denominator is notional, shares or trades.
- **Venue coverage:** evaluate venue relevance by security universe and tape, not only total market share.
- **Regime awareness:** benchmark execution and broker performance separately on peak-activity or auction-heavy sessions.
- **Investigation queue:** use rolling share changes to select venues and dates for deeper order-level analysis and external dialogue.

### Important limitations

The source is daily and aggregated. A TRF record indicates an off-exchange report, not whether the execution occurred at an ATS, wholesaler or another broker-dealer. The data cannot establish spread, fill rate, queue position, price improvement, markout, market impact or latency. HHI describes concentration of reported notional; it is not an execution-quality score. The next step would join internal child-order and fill data to venue, broker, quote and event-calendar features.

### Sources

- [Cboe US Equities Historical Market Volume](https://www.cboe.com/markets/us/equities/market-statistics/historical-market-volume/) — daily shares, notional and trade counts by market centre and tape; Cboe states the data is based on UTDF and CTS consolidated feeds.
- [SEC Regulation NMS](https://www.sec.gov/rules-regulations/2005/06/regulation-nms) — regulatory context for competition, access, quotations and market data.
- [FINRA: Where Do Stocks Trade?](https://www.finra.org/investors/insights/where-do-stocks-trade) — context on exchanges, ATSs and off-exchange reporting.

*Analysis for portfolio demonstration using public data. No employer or client data is used.*
